In [1]:
# Pipeline:
#   Step 1  Load both datasets
#   Step 2  Walk-forward cross-validation (both models)
#   Step 3  Train final models on full dataset
#   Step 4  Calibrate probability outputs
#   Step 5  Save models and metadata
#   Step 6  Sample prediction output

In [2]:
import json
import os
import pickle
import warnings

In [3]:
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
)
from xgboost import XGBClassifier
 
warnings.filterwarnings("ignore")
os.makedirs("saved_models", exist_ok=True)

### Feature and Columns Defs

In [4]:
identity_cols = ["season", "round", "circuit_id", "driver", "team"]
target   = "winner"
 
pre_qual_features = [
    # Block 1 — Team Car Pace (8)
    "team_avg_finish_last_3",
    "team_avg_finish_last_5",
    "team_points_last_3",
    "team_points_this_season",
    "team_championship_position",
    "team_points_gap_to_leader",
    "team_dnf_rate_this_season",
    "team_finish_rate_this_season",
    # Block 2 — Driver Form (8)
    "driver_avg_finish_last_3",
    "driver_avg_finish_last_5",
    "driver_points_this_season",
    "driver_championship_position",
    "driver_points_gap_to_leader",
    "driver_dnf_rate_this_season",
    "driver_finish_rate_this_season",
    "driver_top5_rate_this_season",
    # Block 3 — Historical Qualifying (4)
    "driver_avg_quali_position_last_3",
    "driver_avg_quali_position_last_5",
    "driver_avg_quali_position_this_season",
    "driver_quali_to_finish_delta_last_5",
    # Block 4 — Teammate Comparison (3)
    "avg_quali_gap_to_teammate_this_season",
    "avg_finish_gap_to_teammate_this_season",
    "teammate_head_to_head_ratio_this_season",
    # Block 5 — Driver Circuit History (7)
    "driver_avg_finish_at_circuit_last5",
    "driver_best_finish_at_circuit_last5",
    "driver_podiums_at_circuit_last5",
    "driver_avg_quali_position_at_circuit_last5",
    "driver_dnf_rate_at_circuit",
    "driver_visits_to_circuit",
    "driver_circuit_vs_season_delta",
    # Block 6 — Team Circuit History (6)
    "team_avg_finish_at_circuit_last5",
    "team_wins_at_circuit_last5",
    "team_podium_rate_at_circuit",
    "team_avg_quali_position_at_circuit_last5",
    "team_dnf_rate_at_circuit",
    "team_circuit_vs_season_delta",
    # Block 7 — Circuit Context (4)
    "circuit_type_encoded",
    "circuit_avg_overtakes_last3years",
    "circuit_safety_car_probability",
    "circuit_lap_count",
    # Block 8 — Season Round Context (4)
    "season_round_number",
    "races_remaining_this_season",
    "is_season_opener",
    "seasons_since_regulation_change",
]
 
post_qual_features = pre_qual_features + [
    # Block 9 — This Weekend Qualifying (6)
    "grid_position",
    "gap_to_pole_seconds",
    "gap_to_teammate_quali_this_weekend",
    "quali_position_vs_season_avg",
    "qualifying_session_was_dry",
    "grid_penalty_applied",
    # Block 10 — FP2 Long Run (2)
    "fp2_long_run_avg_gap",
    "fp2_session_was_dry",
    # Block 11 — Race Strategy Signals (3)
    "starting_tyre_compound_encoded",
    "tyre_compound_vs_majority",
    "pit_stop_avg_time_loss_at_circuit",
    # Block 12 — Grid Context (1)
    "drivers_ahead_with_higher_team_pace",
]

### Utilities

In [5]:
# Normalize raw XGBoost scores per race so all 20-22 drivers
# in the same race sum to exactly 1.0.
# Converts raw scores into a valid probability distribution.
def normalize_per_race(df, prob_col):
    return df.groupby(["season", "round"])[prob_col].transform(
        lambda x: x / x.sum() if x.sum() > 0 else x
    )

In [6]:
#  Evaluate model with metrics appropriate for imbalanced binary
#  classification. Top-1 Hit Rate is the primary metric.
def evaluate(model, X, y, df_meta, label):
    raw = model.predict_proba(X)[:, 1]
    meta = df_meta.copy()
    meta["raw_prob"] = raw
    meta["win_prob"] = normalize_per_race(meta, "raw_prob")

    ll = log_loss(y, raw)
    bs = brier_score_loss(y, raw)
    apr = average_precision_score(y, raw)

    top1 = (
        meta.groupby(["season", "round"]).apply(lambda g: g.loc[g["win_prob"].idxmax(), target])
        .mean()
    )

    print(f"\n  [{label}]")
    print(f"    Log Loss       : {ll:.4f}   (lower is better)")
    print(f"    Brier Score    : {bs:.4f}   (lower is better)")
    print(f"    AUC-PR         : {apr:.4f}   (higher is better)")
    print(f"    Top-1 Hit Rate : {top1:.3f}   (target > 0.35 pre / > 0.45 post)")

    return {
        "log_loss":       round(ll,   4),
        "brier_score":    round(bs,   4),
        "auc_pr":         round(apr,  4),
        "top1_hit_rate":  round(top1, 4),
    }

In [7]:
# XGBoost binary classifier configured for small (~2000 row)
#     imbalanced F1 race winner prediction dataset.
 
#     Key decisions:
#       max_depth=3        shallow trees prevent overfitting on small data
#       scale_pos_weight   handles ~19:1 class imbalance
#       min_child_weight=5 regularization — avoids splits on tiny groups
#       eval_metric=aucpr  correct metric for imbalanced binary target
def make_xgb(scale_pos_weight):
    return XGBClassifier(
        objective             = "binary:logistic",
        scale_pos_weight      = scale_pos_weight,
        max_depth             = 3,
        n_estimators          = 300,
        learning_rate         = 0.05,
        subsample             = 0.8,
        colsample_bytree      = 0.8,
        min_child_weight      = 5,
        gamma                 = 1,
        reg_alpha             = 0.1,
        reg_lambda            = 1.0,
        eval_metric           = "aucpr",
        early_stopping_rounds = 25,
        random_state          = 42,
        verbosity             = 0,
    )

### Load Datasets

In [8]:
pre_df   = pd.read_csv("../data/processed/pre_qualifying_dataset.csv")
post_df  = pd.read_csv("../data/processed/post_qualifying_dataset.csv")

In [9]:
seasons = sorted(pre_df["season"].unique().tolist())

# Class imbalance ratio — same for both models (same races, same winner column)
n_pos = int(pre_df[target].sum())
n_neg = int((pre_df[target] == 0).sum())
scale_pos_weight = round(n_neg / max(n_pos, 1))

In [10]:
 # Validate feature columns exist in both datasets
pre_available  = [c for c in pre_qual_features  if c in pre_df.columns]
post_available = [c for c in post_qual_features if c in post_df.columns]
 
pre_missing  = [c for c in pre_qual_features  if c not in pre_df.columns]
post_missing = [c for c in post_qual_features if c not in post_df.columns]

In [11]:
print(f"\n  Pre-qual dataset  : {pre_df.shape}")
print(f"  Post-qual dataset : {post_df.shape}")
print(f"  Seasons           : {seasons}")
print(f"  Winner rows       : {n_pos}")
print(f"  Non-winner rows   : {n_neg}")
print(f"  Class ratio       : 1 : {scale_pos_weight}")
print(f"  scale_pos_weight  : {scale_pos_weight}")
print(f"  Pre-qual features : {len(pre_available)} / {len(pre_qual_features)}")
print(f"  Post-qual features: {len(post_available)} / {len(post_qual_features)}")
 
if pre_missing:
    print(f"Pre-qual missing : {pre_missing}")
if post_missing:
    print(f"Post-qual missing: {post_missing}")


  Pre-qual dataset  : (1838, 50)
  Post-qual dataset : (1838, 62)
  Seasons           : [2022, 2023, 2024, 2025]
  Winner rows       : 92
  Non-winner rows   : 1746
  Class ratio       : 1 : 19
  scale_pos_weight  : 19
  Pre-qual features : 44 / 44
  Post-qual features: 56 / 56


In [12]:
# Last line of defence NaN fill — catches anything that slipped through
# I always fill val with train median, never fit on val data
# This is done inside the CV loop below per fold

### Walk-Forward Cross-Validation

In [13]:
# Walk-forward cross-validation for time-series race data.
# Standard k-fold is WRONG here — it leaks future races into training.
# This function ensures validation always uses data after training data.

def walk_forward_cv(dataset, features_cols, seasons, spw, label):
    fold_metrics = []
    available = [c for c in features_cols if c in dataset.columns]

    for i in range(1, len(seasons)):
        train_seasons = seasons[:i]
        val_season   = seasons[i]

        train = dataset[dataset["season"].isin(train_seasons)].copy()
        val   = dataset[dataset["season"] == val_season].copy()

        if train.empty or val.empty:
            print(f"    Fold {i}: skipped (empty split)")
            continue

        X_train = train[available].copy()
        y_train = train[target].copy()
        X_val   = val[available].copy()
        y_val   = val[target].copy()
        
         # Last line of defence NaN fill
        # Fill val with TRAIN median — never fit on val data
        train_medians = X_train.median()
        X_train = X_train.fillna(train_medians)
        X_val = X_val.fillna(train_medians)

        model = make_xgb(spw)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

        meta = val[["season", "round", target]].copy()
        m = evaluate(model, X_val, y_val, meta, label=f"Fold {i} | Train {train_seasons} -> Val {val_season}")
        
        fold_metrics.append(m)

        if not fold_metrics:
            print(f"  No folds completed for {label}")
            return {}
 
        avg = {k: round(float(np.mean([f[k] for f in fold_metrics])), 4)
            for k in fold_metrics[0]}
    
        print(f"\n  [{label}] Average across {len(fold_metrics)} folds:")
        for k, v in avg.items():
            print(f"    {k}: {v}")
    
        return avg

In [14]:
pre_cv = walk_forward_cv(
    pre_df, pre_qual_features, seasons, scale_pos_weight, "PRE-QUAL CV"
)
post_cv = walk_forward_cv(
    post_df, post_qual_features, seasons, scale_pos_weight, "POST-QUAL CV"
) 


  [Fold 1 | Train [2022] -> Val 2023]
    Log Loss       : 0.2319   (lower is better)
    Brier Score    : 0.0529   (lower is better)
    AUC-PR         : 0.7011   (higher is better)
    Top-1 Hit Rate : 0.773   (target > 0.35 pre / > 0.45 post)

  [PRE-QUAL CV] Average across 1 folds:
    log_loss: 0.2319
    brier_score: 0.0529
    auc_pr: 0.7011
    top1_hit_rate: 0.7727

  [Fold 1 | Train [2022] -> Val 2023]
    Log Loss       : 0.1709   (lower is better)
    Brier Score    : 0.0346   (lower is better)
    AUC-PR         : 0.9258   (higher is better)
    Top-1 Hit Rate : 0.818   (target > 0.35 pre / > 0.45 post)

  [POST-QUAL CV] Average across 1 folds:
    log_loss: 0.1709
    brier_score: 0.0346
    auc_pr: 0.9258
    top1_hit_rate: 0.8182


### Final Model Training

In [15]:
# Train on all seasons except last
# Calibrate on last season — most recent data is best for calibration
train_season = seasons[:-1]
cal_season = seasons[-1]

# Train XGBoost on all-but-last season.
# Calibrate with isotonic regression on last season.
# Calibration converts raw XGBoost scores to proper probabilities.

def train_final_model(dataset, feature_cols, label):
    available  = [c for c in feature_cols if c in dataset.columns]
 
    train_data = dataset[dataset["season"].isin(train_season)].copy()
    cal_data   = dataset[dataset["season"] == cal_season].copy()
 
    X_train = train_data[available].copy()
    y_train = train_data[target].copy()
    X_cal   = cal_data[available].copy()
    y_cal   = cal_data[target].copy()
 
    # Last line of defence NaN fill
    train_medians = X_train.median()
    X_train = X_train.fillna(train_medians)
    X_cal   = X_cal.fillna(train_medians)
 
    model = make_xgb(scale_pos_weight)
    model.early_stopping_rounds = None 
    model.fit(X_train, y_train, verbose=False)
 
    # Calibrate — isotonic regression maps scores to true probabilities
    from sklearn.calibration import CalibratedClassifierCV
    calibrated = CalibratedClassifierCV(
        estimator=model, method="isotonic", cv=5
    )
    calibrated.fit(X_cal, y_cal)
 
    # Evaluate calibrated model on calibration set
    meta = cal_data[["season", "round", target]].copy()
    metrics = evaluate(
        calibrated, X_cal, y_cal, meta,
        label=f"{label} calibrated (cal season {cal_season})"
    )
 
    return model, calibrated, available, train_medians, metrics

In [16]:
pre_model,  pre_cal,  pre_feats,  pre_medians,  pre_final_metrics  = train_final_model(
    pre_df,  pre_qual_features,  "PRE-QUAL MODEL"
)
post_model, post_cal, post_feats, post_medians, post_final_metrics = train_final_model(
    post_df, post_qual_features, "POST-QUAL MODEL"
)
 


  [PRE-QUAL MODEL calibrated (cal season 2025)]
    Log Loss       : 0.0779   (lower is better)
    Brier Score    : 0.0235   (lower is better)
    AUC-PR         : 0.9968   (higher is better)
    Top-1 Hit Rate : 1.000   (target > 0.35 pre / > 0.45 post)

  [POST-QUAL MODEL calibrated (cal season 2025)]
    Log Loss       : 0.0378   (lower is better)
    Brier Score    : 0.0098   (lower is better)
    AUC-PR         : 0.9916   (higher is better)
    Top-1 Hit Rate : 1.000   (target > 0.35 pre / > 0.45 post)


### Feature Importance

In [17]:
def print_feature_importance(model, feature_cols, label, top_n=15):
    importance = model.feature_importances_
    pairs = sorted(
        zip(feature_cols, importance),
        key=lambda x: x[1], reverse=True
    )[:top_n]
    print(f"\n  {label}:")
    for feat, score in pairs:
        print(f"    {feat:<45} {score:.4f}")
 
print_feature_importance(pre_model,  pre_feats,  "Pre-Qualifying Model")
print_feature_importance(post_model, post_feats, "Post-Qualifying Model")


  Pre-Qualifying Model:
    driver_avg_quali_position_last_5              0.2662
    driver_avg_quali_position_last_3              0.0462
    driver_avg_quali_position_this_season         0.0461
    team_avg_finish_last_5                        0.0388
    team_points_gap_to_leader                     0.0371
    driver_points_gap_to_leader                   0.0284
    team_points_last_3                            0.0260
    driver_avg_finish_last_5                      0.0257
    team_championship_position                    0.0249
    team_avg_finish_at_circuit_last5              0.0233
    driver_championship_position                  0.0219
    circuit_safety_car_probability                0.0195
    seasons_since_regulation_change               0.0189
    driver_finish_rate_this_season                0.0185
    avg_finish_gap_to_teammate_this_season        0.0181

  Post-Qualifying Model:
    driver_avg_quali_position_last_5              0.1666
    gap_to_pole_seconds              

### Save Models and Metadata

In [18]:
with open("saved_models/pre_qual_model.pkl",       "wb") as f: pickle.dump(pre_model,   f)

with open("saved_models/post_qual_model.pkl",      "wb") as f: pickle.dump(post_model,  f)

with open("saved_models/pre_qual_calibrated.pkl",  "wb") as f: pickle.dump(pre_cal,     f)

with open("saved_models/post_qual_calibrated.pkl", "wb") as f: pickle.dump(post_cal,    f)

In [19]:
# Save train medians for NaN fill at inference time in Flask API
# API must fill NaN using TRAINING medians, not new data
with open("saved_models/pre_qual_medians.pkl",  "wb") as f: pickle.dump(pre_medians,  f)

with open("saved_models/post_qual_medians.pkl", "wb") as f: pickle.dump(post_medians, f)

In [20]:
# Save all metadata — Flask API loads this at startup
feature_meta = {
    "pre_qual_features":        pre_feats,
    "post_qual_features":       post_feats,
    "identity_cols":            identity_cols,
    "target":                   target,
    "scale_pos_weight":         scale_pos_weight,
    "train_seasons":            train_season,
    "calibration_season":       cal_season,
    "all_seasons":              seasons,
    "pre_qual_cv_metrics":      pre_cv,
    "post_qual_cv_metrics":     post_cv,
    "pre_qual_final_metrics":   pre_final_metrics,
    "post_qual_final_metrics":  post_final_metrics,
}
 
with open("saved_models/feature_columns.json", "w") as f:
    json.dump(feature_meta, f, indent=2)

### SAMPLE PREDICTION ON LAST RACE IN DATASET

In [21]:
# Pick last race in calibration season
last_round   = post_df[post_df["season"] == cal_season]["round"].max()
last_circuit = post_df[
    (post_df["season"] == cal_season) &
    (post_df["round"]  == last_round)
]["circuit_id"].iloc[0]
 
sample = post_df[
    (post_df["season"] == cal_season) &
    (post_df["round"]  == last_round)
].copy()
 
if not sample.empty:
    # Pre-qual prediction
    X_pre_sample = sample[[c for c in pre_feats if c in sample.columns]].copy()
    X_pre_sample = X_pre_sample.fillna(pre_medians)
    pre_raw      = pre_cal.predict_proba(X_pre_sample)[:, 1]
    pre_norm     = pre_raw / pre_raw.sum()
 
    # Post-qual prediction
    X_post_sample = sample[[c for c in post_feats if c in sample.columns]].copy()
    X_post_sample = X_post_sample.fillna(post_medians)
    post_raw      = post_cal.predict_proba(X_post_sample)[:, 1]
    post_norm     = post_raw / post_raw.sum()
 
    sample["pre_qual_prob"]  = pre_norm
    sample["post_qual_prob"] = post_norm
 
    print(f"\n  Race: {last_circuit} {cal_season} (Round {last_round})")
    print(f"\n  {'Driver':<8} {'Team':<22} {'Pre-Qual':>10} {'Post-Qual':>10} {'Grid':>6} {'Result':>8}")
    print(f"  {'-' * 70}")
 
    display = sample[["driver", "team", "pre_qual_prob", "post_qual_prob",
                       "grid_position", target]].copy()
    display  = display.sort_values("post_qual_prob", ascending=False)
 
    for _, r in display.iterrows():
        result = "✓ WIN" if r[target] == 1 else ""
        grid   = int(r["grid_position"]) if pd.notna(r.get("grid_position")) else "-"
        print(
            f"  {str(r['driver']):<8} {str(r['team']):<22} "
            f"{r['pre_qual_prob']:>9.3f}  {r['post_qual_prob']:>9.3f}  "
            f"{str(grid):>5}  {result:>7}"
        )
 
    # Verify probabilities sum to 1
    print(f"\n  Pre-qual probs sum  : {pre_norm.sum():.6f}  (should be 1.0)")
    print(f"  Post-qual probs sum : {post_norm.sum():.6f}  (should be 1.0)")
 


  Race: Yas Island 2025 (Round 24)

  Driver   Team                     Pre-Qual  Post-Qual   Grid   Result
  ----------------------------------------------------------------------
  VER      Red Bull Racing            0.420      0.865      1    ✓ WIN
  NOR      McLaren                    0.119      0.049      2         
  PIA      McLaren                    0.251      0.049      3         
  ANT      Mercedes                   0.027      0.013     14         
  HAM      Ferrari                    0.000      0.013     16         
  LEC      Ferrari                    0.057      0.013      5         
  ALB      Williams                   0.000      0.000     17         
  ALO      Aston Martin               0.000      0.000      6         
  BEA      Haas F1 Team               0.000      0.000     11         
  HAD      Racing Bulls               0.000      0.000      9         
  BOR      Kick Sauber                0.000      0.000      7         
  COL      Alpine                    